# Nemotron Huikang v27 Kaggle-side Repack

This notebook repacks the mounted public adapter into `/kaggle/working/submission.zip`. It does not train, load a base model, or submit to the competition.


In [ ]:
from pathlib import Path
import json
import zipfile
import hashlib
import os
import sys

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
SUBMISSION_ZIP = WORKING_ROOT / "submission.zip"
EXPECTED_ZIP_NAMES = ["adapter_config.json", "adapter_model.safetensors"]


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def candidate_score(candidate):
    path_text = str(candidate["dir"]).lower()
    score = 0
    if "nemotron" in path_text:
        score += 2
    if "adapter" in path_text:
        score += 1
    return score


print("Kaggle-side Nemotron adapter repack")
print("input_root:", INPUT_ROOT)
print("working_root:", WORKING_ROOT)
print("python:", sys.version)

if not INPUT_ROOT.exists():
    raise SystemExit("ERROR: /kaggle/input does not exist. Check notebook inputs.")

configs = sorted(INPUT_ROOT.rglob("adapter_config.json"))
candidates = []
for config_path in configs:
    model_path = config_path.parent / "adapter_model.safetensors"
    candidates.append(
        {
            "dir": config_path.parent,
            "config": config_path,
            "model": model_path,
            "model_exists": model_path.exists(),
        }
    )

print("adapter candidates:")
for idx, candidate in enumerate(candidates, 1):
    print(
        idx,
        "dir=",
        candidate["dir"],
        "model_exists=",
        candidate["model_exists"],
    )

valid_candidates = [candidate for candidate in candidates if candidate["model_exists"]]
if not valid_candidates:
    raise SystemExit("ERROR: no adapter_config.json with sibling adapter_model.safetensors was found.")

selected = sorted(valid_candidates, key=lambda item: (-candidate_score(item), str(item["dir"])))[0]
config_path = selected["config"]
model_path = selected["model"]
print("selected_adapter_dir:", selected["dir"])

config = json.loads(config_path.read_text(encoding="utf-8"))
peft_type = config.get("peft_type")
rank = config.get("r")
base_model = config.get("base_model_name_or_path")
print("peft_type:", peft_type)
print("r:", rank)
print("base_model_name_or_path:", base_model)

if rank is not None:
    try:
        rank_int = int(rank)
    except Exception as exc:
        raise SystemExit(f"ERROR: adapter rank r is not an integer: {rank!r}") from exc
    if rank_int > 32:
        raise SystemExit(f"ERROR: adapter rank r={rank_int} exceeds competition limit 32.")
else:
    print("WARNING: adapter rank r is missing; zip will be built after structural file checks only.")

WORKING_ROOT.mkdir(parents=True, exist_ok=True)
if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_STORED) as zf:
    zf.write(config_path, arcname="adapter_config.json")
    zf.write(model_path, arcname="adapter_model.safetensors")

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    names = sorted(zf.namelist())
print("zip_namelist:", names)
if names != EXPECTED_ZIP_NAMES:
    raise SystemExit(f"ERROR: invalid zip root contents: {names!r}")

size_bytes = SUBMISSION_ZIP.stat().st_size
sha256 = sha256_file(SUBMISSION_ZIP)
print("submission_zip:", SUBMISSION_ZIP)
print("submission_zip_size_bytes:", size_bytes)
print("submission_zip_sha256:", sha256)
print("OK: /kaggle/working/submission.zip is ready.")
